# reflective-data-catalog — Example

A unified interface for SAI (Stratospheric Aerosol Injection) climate model data in cloud storage.

```bash
pip install reflective-data-catalog
```

Public (ARISE) sources need no credentials. Reflective hub sources need AWS credentials — set `HUB = True` below if you have them. See the README's credentials matrix.

In [1]:
from reflective_data_catalog import ReflectiveCatalog

catalog = ReflectiveCatalog()
HUB = False  # set True if you have Reflective hub credentials

## Browse the catalog

`list_sources()` returns structured records (and prints a listing unless `verbose=False`).

In [2]:
records = catalog.list_sources(verbose=False)
[(r['name'], r['driver'], r['stability']) for r in records][:8]

[('arise_15_cesm2_waccm_ssp245', 'netcdf', 'stable'),
 ('arise_sai_15', 'netcdf', 'stable'),
 ('cesm2_waccm6_gauss_historical', 'zarr', 'experimental'),
 ('cesm2_waccm_g6_1p5k_hilla', 'zarr', 'stable'),
 ('cesm2_waccm_g6_1p5k_sai', 'zarr', 'stable'),
 ('cesm2_waccm_historical', 'netcdf', 'stable'),
 ('cesm2_waccm_ssp245', 'zarr', 'stable'),
 ('e3smv3_g6_1p5k_hilla', 'zarr', 'stable')]

In [3]:
catalog.search(term='SAI', verbose=False)[:5]

[{'name': 'arise_15_cesm2_waccm_ssp245',
  'kind': 'entry',
  'driver': 'netcdf',
  'description': 'CESM2-WACCM SSP2-4.5 (the ARISE-SAI reference scenario runs, public NCAR bucket)'},
 {'name': 'arise_sai_15',
  'kind': 'entry',
  'driver': 'netcdf',
  'description': 'CESM2-WACCM ARISE-SAI: SAI at 30°N/S and 15°N/S (21km) with multi-objective feedback control'},
 {'name': 'cesm2_waccm6_gauss_historical',
  'kind': 'entry',
  'driver': 'zarr',
  'description': 'CESM2-WACCM GAUSS: Various SAI simulations with multiple injection strategies and temperature targets'},
 {'name': 'cesm2_waccm_g6_1p5k_hilla',
  'kind': 'entry',
  'driver': 'zarr',
  'description': 'CESM2-WACCM G6-1.5K-HiLLA (Zarr): SAI with 60°N/S injection at 15km to hold global mean temperature to 1.5K'},
 {'name': 'cesm2_waccm_g6_1p5k_sai',
  'kind': 'entry',
  'driver': 'zarr',
  'description': 'CESM2-WACCM G6-1.5K-SAI: SAI 30°N/S injection at 21km to hold global mean temperature to 1.5K'}]

In [4]:
catalog.get_parameters('ukesm1_arise_sai')

{'name': 'ukesm1_arise_sai',
 'driver': 'netcdf',
 'description': 'UKESM1.0 ARISE-SAI: SAI at 30°N/S and 15°N/S (21km) with multi-objective feedback control',
 'parameters': {'variable': {'description': 'Climate variable name',
   'type': 'str',
   'default': 'tas'},
  'ensemble': {'description': 'Ensemble member identifier',
   'type': 'str',
   'default': 'r1i1p1f2',
   'allowed': ['r1i1p1f2', 'r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r8i1p1f2']},
  'table': {'description': 'CMIP6 table ID (e.g., day, Amon, Omon)',
   'type': 'str',
   'default': 'day'}}}

## Discover what's really in the bucket

Discovery lists the live bucket — no credentials needed for public sources.

In [5]:
src = catalog.ukesm1_arise_sai()
print('members:', src.list_ensembles())
print('tables:', src.list_tables()[:10])

members: ['r1i1p1f2', 'r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r8i1p1f2']
tables: ['3hr', '6hrPlevPt', 'AERmon', 'AERmonZ', 'Amon', 'CFday', 'CFmon', 'E3hr', 'Eday', 'EdayZ']


## Load public data (anonymous)

`to_dask()` opens lazily; `read()` loads into memory.

In [6]:
ds = catalog.ukesm1_arise_sai(variable='tas', table='Amon').to_dask()
ds

/Users/john/Projects/reflective-data-catalog/src/reflective_data_catalog/loader.py:395: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(handles, **mf_kwargs)


<xarray.Dataset> Size: 50MB
Dimensions:    (time: 432, bnds: 2, lat: 144, lon: 192)
Coordinates:
  * time       (time) object 3kB 2035-01-16 00:00:00 ... 2070-12-16 00:00:00
  * lat        (lat) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
  * lon        (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
    height     float64 8B 1.5
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object 7kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    lat_bnds   (time, lat, bnds) float64 995kB dask.array<chunksize=(180, 144, 2), meta=np.ndarray>
    lon_bnds   (time, lon, bnds) float64 1MB dask.array<chunksize=(180, 192, 2), meta=np.ndarray>
    tas        (time, lat, lon) float32 48MB dask.array<chunksize=(1, 144, 192), meta=np.ndarray>
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ARISE
    branch_method:          standard
    branch_time_in_child:   66600.0
    branch_time_in_parent:  66600.0
    creation_date:          2022-03-25T12:19:24Z
    ...                     ...
    variable_id:            tas
    variable_name:          tas
    variant_label:          r1i1p1f2
    license:                ARISE data produced by MOHC is licensed under the...
    cmor_version:           3.6.1
    tracking_id:            ARISE/c13441f9-ff43-4d2e-add0-608134dd3501

## Public Zarr stores on Cloudflare R2

The CESM2-WACCM and MIROC-ES2H sources are grouped public Zarr stores — no credentials needed. Tables and realms are Zarr groups; the ensemble is a dataset dimension (`ensemble='all'` keeps every member).

In [7]:
ds = catalog.cesm2_waccm_g6_1p5k_hilla(table='Amon', realm='atmos_2d', ensemble='r1').to_dask()
print('CESM HiLLA (public R2):', dict(ds.sizes))
miroc = catalog.miroc_es2h_g6_1p5k_sai(ensemble='all').to_dask()
print('MIROC SAI full ensemble:', dict(miroc.sizes))

if HUB:  # UKESM/E3SMv3 stores live on the Reflective hub (AWS credentials)
    ds = catalog.ukesm1_g6_1p5k_hilla(variable='tas', table='Aday').to_dask()
    display(ds)

CESM HiLLA (public R2): {'time': 600, 'lat': 192, 'lon': 288, 'ilev': 71, 'lev': 70, 'bnds': 2, 'zlon': 1}


MIROC SAI full ensemble: {'member': 10, 'time': 600, 'lat': 128, 'lon': 256}


## Typos and old vocabulary error loudly

Unknown keywords raise `TypeError` (pre-1.0 they silently loaded default data); removed kwargs carry migration guidance.

In [8]:
try:
    catalog.cesm2_waccm_ssp245(varaible='SALT')
except TypeError as e:
    print(e)
try:
    catalog.ukesm1_g6_1p5k_hilla(time='AERmon')
except TypeError as e:
    print(e)

cesm2_waccm_ssp245: unknown parameter(s) ['varaible']. Valid parameters: ['ensemble', 'realm', 'table'] (aliases accepted: ['ensemble_member', 'member_id', 'table_id'])
ukesm1_g6_1p5k_hilla: unknown parameter(s) ['time']. Valid parameters: ['ensemble', 'table', 'variable'] (aliases accepted: ['ensemble_member', 'member_id', 'table_id', 'variable_id']). Migration note — 'time': removed with the UKESM Zarr switch; the stream/time split collapses into CMOR tables — error redirects to the migration guide


## Google Cloud CMIP6 / GeoMIP (optional extra)

Requires `pip install "reflective-data-catalog[esm]"`.

In [9]:
try:
    esm = catalog.esm
    print(type(esm).__name__, 'ready')
except ImportError as e:
    print(e)

ESMCatalog ready
